# Model quality review

Review diagnostics, unnamed elements, repeated names, and empty definitions.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

Loaded 42 SysML files from /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model


In [2]:
from IPython.display import display

usages = list(model.elements(syside.Usage, include_subtypes=True))
definitions = list(model.elements(syside.Definition, include_subtypes=True))
semantic_elements = usages + definitions

unnamed = [item for item in semantic_elements if not (item.name or item.declared_name)]
by_name = defaultdict(list)
for item in semantic_elements:
    value = item.qualified_name or item.name or item.declared_name
    name = str(value) if value else None
    if name:
        by_name[name].append(item)
duplicates = [(name, len(items), sorted({type(item).__name__ for item in items})) for name, items in by_name.items() if len(items) > 1]
empty_definitions = [
    item for item in definitions
    if not any(isinstance(child, (syside.Usage, syside.Definition)) for child in item.owned_elements)
]

display({
    'parser/semantic errors': len(list(diagnostics.errors)),
    'warnings': len(list(diagnostics.warnings)),
    'unnamed usages or definitions': len(unnamed),
    'repeated qualified names': len(duplicates),
    'definitions with no owned usages/definitions': len(empty_definitions),
})

print('Repeated qualified names (first 30)')
display(sorted(duplicates, key=lambda item: item[1], reverse=True)[:30])

print('Empty definition candidates (first 30)')
[(type(item).__name__, item.qualified_name or item.name) for item in empty_definitions[:30]]

{'parser/semantic errors': 8009,
 'warnings': 0,
 'unnamed usages or definitions': 5978,
 'repeated qualified names': 5,
 'definitions with no owned usages/definitions': 1}

Repeated qualified names (first 30)


[('sensorStatus', 2, ['ReferenceUsage']),
 ('alarmSignal', 2, ['ReferenceUsage']),
 ('authorizedBolus', 2, ['ReferenceUsage']),
 ('flowCommand', 2, ['ReferenceUsage']),
 ('actuation', 2, ['ReferenceUsage'])]

Empty definition candidates (first 30)


[('ConjugatedPortDefinition',
  _syside.core.QualifiedName(['memo_examples_gpca_pump_model_catalog_gpca_interfaces', 'SensorInputPort', '~SensorInputPort']))]

In [3]:
print('First diagnostics to investigate')
for diagnostic in list(diagnostics.errors)[:20] + list(diagnostics.warnings)[:20]:
    print(diagnostic)

First diagnostics to investigate
/Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model/catalog/gpca_cybersecurity.sysml:159:5: error (connector-related-features): A concrete connector must have at least two related elements
/Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model/catalog/gpca_interfaces.sysml:19:5: error (connector-related-features): A concrete connector must have at least two related elements
/Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model/catalog/gpca_interfaces.sysml:27:5: error (connector-related-features): A concrete connector must have at least two related elements
/Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model/catalog/gpca_interfaces.sysml:35:5: error (connector-related-features): A concrete connector must have at least two related elements
/Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model/catalog/gpca_interfaces.sysml:43:5: error (connector-related-features): A concrete connecto